# Module 2: Prompting Strategy Playground

Compare zero-shot, one-shot, few-shot, system-prompted, and delimited prompting
on the same tasks. See how strategy choice affects output format, consistency, and cost.

## Configuration

Set your provider and model below. The rest of the notebook adapts automatically.

In [ ]:
import sys
sys.path.insert(0, '..')

from utils.llm_client import call_llm
import pandas as pd

# === CONFIGURE THIS CELL ===
PROVIDER = "openai"        # "openai", "anthropic", or "ollama"
MODEL = "gpt-4o"           # or "claude-sonnet-4-20250514", "llama3"
# ===========================

print(f"Provider: {PROVIDER} | Model: {MODEL}")

---
## 1. Zero-Shot vs Few-Shot: Sentiment Classification

The most direct comparison: same task, same model, different number of examples.

In [ ]:
TEST_INPUT = "The product works fine but the shipping was slow."

# --- Zero-Shot ---
zero_shot_prompt = f"""Classify the following text as positive, negative, or neutral:

Text: {TEST_INPUT}
Sentiment:"""

zero_shot_response = call_llm(
    zero_shot_prompt,
    provider=PROVIDER,
    model=MODEL,
    temperature=0.0,
    max_tokens=50,
)

print("=== ZERO-SHOT ===")
print(f"Prompt: {zero_shot_prompt}")
print(f"Output: {zero_shot_response}")

In [ ]:
# --- One-Shot ---
one_shot_prompt = f"""Classify the following text as positive, negative, or neutral:

Text: "I love this product!"
Sentiment: Positive

Text: {TEST_INPUT}
Sentiment:"""

one_shot_response = call_llm(
    one_shot_prompt,
    provider=PROVIDER,
    model=MODEL,
    temperature=0.0,
    max_tokens=50,
)

print("=== ONE-SHOT ===")
print(f"Prompt: {one_shot_prompt}")
print(f"Output: {one_shot_response}")

In [ ]:
# --- Few-Shot (3 examples) ---
few_shot_prompt = f"""Classify the following text as positive, negative, or neutral:

Text: "I love this product!"
Sentiment: Positive

Text: "Terrible experience, would not recommend."
Sentiment: Negative

Text: "It's okay, nothing special."
Sentiment: Neutral

Text: {TEST_INPUT}
Sentiment:"""

few_shot_response = call_llm(
    few_shot_prompt,
    provider=PROVIDER,
    model=MODEL,
    temperature=0.0,
    max_tokens=50,
)

print("=== FEW-SHOT (3 examples) ===")
print(f"Prompt: {few_shot_prompt}")
print(f"Output: {few_shot_response}")

In [ ]:
# --- Comparison Table ---
sentiment_results = pd.DataFrame([
    {"strategy": "Zero-Shot", "examples": 0, "output": zero_shot_response.strip()},
    {"strategy": "One-Shot", "examples": 1, "output": one_shot_response.strip()},
    {"strategy": "Few-Shot", "examples": 3, "output": few_shot_response.strip()},
])

print("\n=== SENTIMENT CLASSIFICATION COMPARISON ===")
print(f"Input: {TEST_INPUT}\n")
print(sentiment_results.to_string(index=False))

token_estimate = len(few_shot_prompt.split()) - len(zero_shot_prompt.split())
print(f"\nFew-shot used ~{token_estimate} more input tokens than zero-shot.")

---
## 2. System Instructions: Persona Impact

Same question, different system instructions. See how the persona shapes the response.

In [ ]:
QUESTION = "What is a REST API?"

personas = {
    "Generic": "You are a helpful assistant.",
    "Junior Dev Teacher": "You explain concepts to junior developers. Use simple analogies and avoid jargon.",
    "Senior Architect": "You are a principal software architect. Be precise, mention tradeoffs, and reference real-world patterns.",
    "Security Engineer": "You are a security engineer. Focus on authentication, authorization, and common vulnerabilities of REST APIs.",
}

persona_results = []
for name, system_prompt in personas.items():
    response = call_llm(
        QUESTION,
        provider=PROVIDER,
        model=MODEL,
        system_prompt=system_prompt,
        temperature=0.3,
        max_tokens=200,
    )
    persona_results.append({"persona": name, "system_prompt": system_prompt, "response": response})
    print(f"\n=== {name.upper()} ===")
    print(f"System: {system_prompt}")
    print(f"Output: {response[:250]}...")

print("\n\n=== PERSONA COMPARISON ===")
persona_df = pd.DataFrame(persona_results)
persona_df["response_preview"] = persona_df["response"].str[:150] + "..."
persona_df[["persona", "response_preview"]]

---
## 3. Delimiter Formats: Structuring Inputs

Compare how different delimiter styles affect the model's ability to parse instructions vs data.

In [ ]:
ARTICLE = """Climate change is causing sea levels to rise at an accelerating rate. 
Since 1900, global sea levels have risen approximately 8 inches, with the rate 
doubling in the last two decades. Coastal cities like Miami, Jakarta, and Mumbai 
face increasing flooding risk. Scientists project 1-3 feet of additional rise 
by 2100 depending on emissions scenarios."""

# --- No delimiters (control) ---
no_delim_prompt = f"Summarize the following article about climate change in one sentence. {ARTICLE}"

# --- Markdown headers ---
md_delim_prompt = f"""### Instructions
Summarize the following article in one sentence.

### Article
{ARTICLE}"""

# --- XML tags ---
xml_delim_prompt = f"""<instructions>
Summarize the following article in one sentence.
</instructions>

<article>
{ARTICLE}
</article>"""

delim_results = []
for name, prompt in [("No Delimiters", no_delim_prompt), ("Markdown ###", md_delim_prompt), ("XML Tags", xml_delim_prompt)]:
    response = call_llm(
        prompt,
        provider=PROVIDER,
        model=MODEL,
        temperature=0.0,
        max_tokens=100,
    )
    delim_results.append({"format": name, "output": response.strip()})
    print(f"\n=== {name.upper()} ===")
    print(f"Output: {response.strip()}")

print("\n\n=== DELIMITER COMPARISON ===")
pd.DataFrame(delim_results)

---
## 4. Combined Technique: Production Prompt

A production-grade prompt combining system instruction + few-shot + delimiters on a structured extraction task.

In [ ]:
# --- Production-style prompt: extract structured data from support tickets ---
system = "You are a support ticket parser. Output ONLY valid JSON. No explanation."

production_prompt = """### Task
Extract category, priority, and customer sentiment from the support ticket.

### Category options: Billing, Technical, Account, Feature Request
### Priority options: low, medium, high, urgent
### Sentiment options: positive, neutral, negative, angry

### Examples

<example>
<input>I was charged twice this month and nobody is responding to my emails.</input>
<output>{"category": "Billing", "priority": "high", "sentiment": "angry"}</output>
</example>

<example>
<input>Can you add dark mode to the mobile app?</input>
<output>{"category": "Feature Request", "priority": "low", "sentiment": "neutral"}</output>
</example>

### New Ticket
<input>The app keeps crashing when I try to upload large files. This is blocking our team from submitting reports.</input>
<output>"""

prod_response = call_llm(
    production_prompt,
    provider=PROVIDER,
    model=MODEL,
    system_prompt=system,
    temperature=0.0,
    max_tokens=100,
)

print("=== PRODUCTION PROMPT OUTPUT ===")
print(f"System: {system}")
print(f"\nPrompt:\n{production_prompt}")
print(f"\nOutput: {prod_response.strip()}")

# Validate it's parseable JSON
import json
try:
    parsed = json.loads(prod_response.strip())
    print(f"\nParsed successfully: {parsed}")
except json.JSONDecodeError as e:
    print(f"\nWARNING: Output is not valid JSON: {e}")

---
## 5. Consistency Test

Run the same zero-shot prompt 3 times at temperature=0. Then run the same few-shot prompt 3 times. Compare consistency.

In [ ]:
CONSISTENCY_INPUT = "The food was great but the service was painfully slow."

zero_template = f"""Classify sentiment as positive, negative, or neutral:

Text: {CONSISTENCY_INPUT}
Sentiment:"""

few_template = f"""Classify sentiment as positive, negative, or neutral:

Text: "I love this place!"
Sentiment: Positive

Text: "Worst experience ever."
Sentiment: Negative

Text: "It was fine."
Sentiment: Neutral

Text: {CONSISTENCY_INPUT}
Sentiment:"""

zero_outputs = []
few_outputs = []

for i in range(3):
    z = call_llm(zero_template, provider=PROVIDER, model=MODEL, temperature=0.0, max_tokens=20)
    f = call_llm(few_template, provider=PROVIDER, model=MODEL, temperature=0.0, max_tokens=20)
    zero_outputs.append(z.strip())
    few_outputs.append(f.strip())

consistency_df = pd.DataFrame({
    "Run": [1, 2, 3],
    "Zero-Shot": zero_outputs,
    "Few-Shot": few_outputs,
})

print("=== CONSISTENCY TEST ===")
print(f"Input: {CONSISTENCY_INPUT}\n")
print(consistency_df.to_string(index=False))
print(f"\nZero-shot unique outputs: {len(set(zero_outputs))}")
print(f"Few-shot unique outputs: {len(set(few_outputs))}")

---
## Summary

Key takeaways from this notebook:

1. **Zero-shot** is cheapest and works for straightforward tasks. Start here.
2. **One-shot/few-shot** improves format consistency — the model follows the demonstrated pattern rather than guessing.
3. **System instructions** shape persona and tone. Same question, different persona = dramatically different output.
4. **Delimiters** make instruction-data boundaries explicit. XML tags and markdown headers both work; pick one and stay consistent.
5. **Production prompts combine all techniques**: system instruction + few-shot + delimiters = reliable, parseable output.
6. **Consistency test** shows few-shot produces more repeatable labels than zero-shot on ambiguous inputs.

Next: [Module 3 - Advanced Reasoning & Logic Techniques](../03-reasoning-and-logic/README.md)